# Homework 2: Step-by-Step Solution

This notebook follows the homework as a sequence of local question-and-answer blocks:

1. Read the original question.
2. Explore the relevant data.
3. Write the answer immediately below that exploration.

It assumes you are comfortable with programming, but new to genomics. The goal is to show how each answer is derived from the metadata, FASTQ files, FastQC reports, and the `hg38` reference genome.

Expected data location in WSL:

```bash
~/compgenomicsws_hw2
```

If your files are somewhere else, update `DATA_DIR` in the first code cell.

## 0. Big picture for a computer science student

The assignment is mostly a file-format and data-inspection exercise. You do not need to understand cancer biology deeply to solve it. Think of it as parsing structured files and explaining what the fields mean.

The story is:

1. We have two DNA sequencing samples from one patient: one tumor sample and one nearby normal sample.
2. The sequencing machine produced compressed FASTQ files. A FASTQ file is like a large text log where every read uses exactly four lines.
3. The reads are paired-end, meaning each DNA fragment was read from two sides: `R1` and `R2`.
4. FastQC checks whether the raw sequencing data looks technically usable.
5. `hg38` is the reference human genome. We inspect it as a FASTA file, another simple text format.

Useful translations:

- **Sample**: one biological source that was sequenced, like one tumor biopsy.
- **Read**: one short DNA string emitted by the sequencer, similar to one record in a huge data file.
- **Base**: one DNA character: `A`, `C`, `G`, or `T`.
- **bp**: base pairs. In this homework, a `100 bp` read means a DNA string of length 100.
- **Reference genome**: the standard coordinate system used to describe human DNA positions.

## 1. Setup and Files From the README

The README tells us to use:

- `Metadata_18.csv` for sample metadata.
- Four paired-end FASTQ files: normal `R1/R2` and tumor `R1/R2`.
- FastQC for raw-read quality control.
- `hg38.fa.gz` as the human reference genome.

We will use Python for reproducible parsing and shell commands only where they mirror the original assignment workflow. The FASTQ and FASTA files are gzip-compressed, so Python's `gzip` module can read them directly without manually unzipping huge files.

What this setup cell does:

- Points Python at the assignment folder and downloaded data folder.
- Gives friendly names to the four FASTQ files.
- Checks whether the expected files exist before we spend time parsing them.

In [1]:
from pathlib import Path
import os
import gzip
import csv
import zipfile
import subprocess
import statistics
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")
import pandas as pd
import plotly.express as px
from IPython.display import display

DATA_DIR = Path.home() / "compgenomicsws_hw2"

# VS Code/Jupyter may start either in this assignment folder or at the repo root.
# This small check makes the notebook work in both cases.
ASSIGNMENT_DIR = Path.cwd()
if not (ASSIGNMENT_DIR / "Metadata_18.csv").exists():
    candidate = ASSIGNMENT_DIR / "Lesson3[IntroNGS]+Assignment2"
    if candidate.exists():
        ASSIGNMENT_DIR = candidate

METADATA = ASSIGNMENT_DIR / "Metadata_18.csv"

FASTQ_FILES = {
    "normal_R1": DATA_DIR / "ERR4833597_1.fastq.gz",
    "normal_R2": DATA_DIR / "ERR4833597_2.fastq.gz",
    "tumor_R1": DATA_DIR / "ERR4833621_1.fastq.gz",
    "tumor_R2": DATA_DIR / "ERR4833621_2.fastq.gz",
}

HG38 = DATA_DIR / "hg38.fa.gz"
FASTQC_DIR = DATA_DIR / "fastqc"

print(f"Assignment directory: {ASSIGNMENT_DIR}")
print(f"Data directory:       {DATA_DIR}")
print("\nFASTQ files:")
for label, path in FASTQ_FILES.items():
    print(f"{label:10s} {path} exists={path.exists()}")
print(f"\nhg38 exists={HG38.exists()} path={HG38}")



Assignment directory: /mnt/c/Users/orgrd/workspace/repos/CompGenomicsWS/Lesson3[IntroNGS]+Assignment2
Data directory:       /home/orgr/compgenomicsws_hw2

FASTQ files:
normal_R1  /home/orgr/compgenomicsws_hw2/ERR4833597_1.fastq.gz exists=True
normal_R2  /home/orgr/compgenomicsws_hw2/ERR4833597_2.fastq.gz exists=True
tumor_R1   /home/orgr/compgenomicsws_hw2/ERR4833621_1.fastq.gz exists=True
tumor_R2   /home/orgr/compgenomicsws_hw2/ERR4833621_2.fastq.gz exists=True

hg38 exists=True path=/home/orgr/compgenomicsws_hw2/hg38.fa.gz


## 2. Metadata Questions

We start with `Metadata_18.csv`, because it tells us what the downloaded files actually represent.

### Question 1

> The file `Metadata_18.csv` contains additional meta information about the files we are downloading. What does `neoplasm` and `normal tissue adjacent to neoplasm` mean?

Before answering, we need to inspect the metadata and translate the biological labels into plain language.

The metadata file tells us that there are two samples from the same patient:

- one sample from the tumor area
- one sample from nearby tissue that is not considered tumor

Why compare them? A person's DNA already differs from the generic reference genome in many normal inherited ways. If we only looked at the tumor, we would not know whether a difference is just part of this person's normal DNA or something specific to the tumor. The nearby non-tumor sample acts like the patient's personal baseline/control.

Programming analogy: comparing tumor DNA only to `hg38` is like diffing a modified config file against a generic template. Comparing tumor DNA to the same patient's normal DNA is like diffing against the original version from that same machine.

Important vocabulary:

- **Neoplasm**: literally an abnormal new growth of tissue. In this homework, treat it as the tumor biopsy/sample.
- **Normal tissue adjacent to neoplasm**: tissue collected near the tumor, from the same patient, that is not classified as tumor. It is not "normal" in the philosophical sense; it just means "the comparison/control tissue for this patient".
- **Germline genotype**: inherited DNA, present in normal cells.
- **Somatic genotype**: DNA changes found in tumor cells, acquired during life rather than inherited.

In [2]:
# The CSV contains duplicate column names, so csv.DictReader is awkward here.
# We read rows manually and display only the columns that answer the homework questions.
with METADATA.open(newline="") as fh:
    reader = csv.reader(fh)
    header = next(reader)
    rows = list(reader)

interesting_columns = [
    "Characteristics[individual]",
    "Characteristics[organism part]",
    "Characteristics[sampling site]",
    "Characteristics[disease]",
    "Characteristics[genotype]",
    "Comment[LIBRARY_LAYOUT]",
    "Comment[LIBRARY_STRATEGY]",
    "Scan Name",
    "Comment[FASTQ_URI]",
]

indices = [header.index(col) for col in interesting_columns]

for row in rows:
    print("-" * 80)
    for col, idx in zip(interesting_columns, indices):
        print(f"{col:35s}: {row[idx]}")

--------------------------------------------------------------------------------
Characteristics[individual]        : AD0699_18
Characteristics[organism part]     : uterine cervix
Characteristics[sampling site]     : normal tissue adjacent to neoplasm
Characteristics[disease]           : cervical cancer
Characteristics[genotype]          : germline genotype
Comment[LIBRARY_LAYOUT]            : PAIRED
Comment[LIBRARY_STRATEGY]          : WXS
Scan Name                          : AD0699_18N_R1.fastq.gz
Comment[FASTQ_URI]                 : ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR483/007/ERR4833597/ERR4833597_1.fastq.gz
--------------------------------------------------------------------------------
Characteristics[individual]        : AD0699_18
Characteristics[organism part]     : uterine cervix
Characteristics[sampling site]     : normal tissue adjacent to neoplasm
Characteristics[disease]           : cervical cancer
Characteristics[genotype]          : germline genotype
Comment[LIBRARY_LAY

### Data exploration: what is in the metadata?

Before answering, it helps to ask: "What are the rows, and which columns tell me what each FASTQ file represents?"

This exploration cell turns the important metadata columns into a compact table-like view. It also counts sample types and library strategies. The goal is to make the biological labels feel like normal categorical data.

In [3]:
# Build small dictionaries for the columns we care about.
metadata_records = []
for row in rows:
    metadata_records.append({col: row[idx] for col, idx in zip(interesting_columns, indices)})

for i, record in enumerate(metadata_records, start=1):
    print(f"Sample {i}")
    print(f"  run/accession: {record['Scan Name']}")
    print(f"  tissue label:  {record['Characteristics[organism part]']}")
    print(f"  genotype:      {record['Characteristics[genotype]']}")
    print(f"  layout:        {record['Comment[LIBRARY_LAYOUT]']}")
    print(f"  strategy:      {record['Comment[LIBRARY_STRATEGY]']}")
    print()

# Simple categorical counts. This is the metadata as ordinary tabular data.
def count_values(records, key):
    counts = {}
    for record in records:
        counts[record[key]] = counts.get(record[key], 0) + 1
    return counts

summary_rows = []
for label, key in [
    ("Tissue labels", "Characteristics[organism part]"),
    ("Genotypes", "Characteristics[genotype]"),
    ("Library strategies", "Comment[LIBRARY_STRATEGY]"),
    ("Library layouts", "Comment[LIBRARY_LAYOUT]"),
]:
    counts = count_values(metadata_records, key)
    print(label)
    for value, count in counts.items():
        print(f"  {value}: {count}")
        summary_rows.append({"category": label, "value": value, "count": count})
    print()

metadata_summary_df = pd.DataFrame(summary_rows)
fig = px.bar(
    metadata_summary_df,
    x="value",
    y="count",
    color="category",
    facet_col="category",
    facet_col_wrap=2,
    title="Metadata categories used to understand the samples",
    labels={"value": "metadata value", "count": "number of rows"},
)
fig.update_xaxes(matches=None, tickangle=25)
fig.update_yaxes(dtick=1)
fig.show()

Sample 1
  run/accession: AD0699_18N_R1.fastq.gz
  tissue label:  uterine cervix
  genotype:      germline genotype
  layout:        PAIRED
  strategy:      WXS

Sample 2
  run/accession: AD0699_18N_R2.fastq.gz
  tissue label:  uterine cervix
  genotype:      germline genotype
  layout:        PAIRED
  strategy:      WXS

Sample 3
  run/accession: AD0729_18T_R1.fastq.gz
  tissue label:  uterine cervix
  genotype:      somatic genotype
  layout:        PAIRED
  strategy:      WXS

Sample 4
  run/accession: AD0729_18T_R2.fastq.gz
  tissue label:  uterine cervix
  genotype:      somatic genotype
  layout:        PAIRED
  strategy:      WXS

Tissue labels
  uterine cervix: 4

Genotypes
  germline genotype: 2
  somatic genotype: 2

Library strategies
  WXS: 4

Library layouts
  PAIRED: 4



### Answer 1

In this metadata, `neoplasm` means the sample taken from the tumor tissue. A neoplasm is an abnormal tissue growth; for this assignment, you can read it as "the tumor sample".

`normal tissue adjacent to neoplasm` means tissue taken from near the tumor, from the same patient, that is not labeled as tumor. It is used as the patient's own baseline/control sample.

This matters because every person has many normal DNA differences. By comparing tumor vs same-patient normal, we can later ask: "which DNA differences are specific to the tumor?"

Simple mapping:

- `ERR4833621` / `AD0729_18T` = tumor sample
- `ERR4833597` / `AD0699_18N` = matched normal/control sample

### Question 2

> The 23rd column (`LIBRARY_STRATEGY`) is marked as `WXS`. What is Whole Exome Sequencing (`WXS`/`WES`)? What would be the FASTQ file sizes if we did Whole Genome Sequencing (`WGS`) instead?

### Answer 2

`WXS`/`WES` means whole-exome sequencing. It targets exons, not the whole genome. Exons are the protein-coding parts of genes, so they are a much smaller target than all human DNA.

If this were whole-genome sequencing (`WGS`), the FASTQ files would usually be much larger: typically tens of GB per compressed FASTQ file, often around `30-100+ GB` per file depending on coverage, read length, and compression.

## 3. FASTQ Headers

### Question 3

> Once the downloads have completed start by simply looking at each file and making sure the data format is OK. Make sure the read titles make sense and match between `R1` and `R2`.
>
> What is the difference between the header of the first read in `R1` and the header of the first read in `R2`?

FASTQ records always use four lines. This is the most important file-format fact in the assignment:

1. Header line, beginning with `@`
2. DNA sequence
3. Separator line, usually `+`
4. Quality string

For paired-end reads, `R1` and `R2` should have matching read IDs, but one ends in `/1` and the other ends in `/2`.

Mental model: `R1` and `R2` are not two different patients or two different experiments. They are two reads from opposite ends of the same original DNA fragment.

In [4]:
def first_fastq_record(path):
    with gzip.open(path, "rt") as fh:
        return [next(fh).rstrip() for _ in range(4)]

for label in ["normal_R1", "normal_R2", "tumor_R1", "tumor_R2"]:
    record = first_fastq_record(FASTQ_FILES[label])
    print("=" * 80)
    print(label)
    print("\n".join(record))

normal_R1
@ERR4833597.1 HISEQ:451:C78DHACXX:7:1101:10000:14561/1
GGCGGCGGCGGCGGCGGCGGCGGCGGCGGCCGGGGCTGGTGGGGGAGGGGGGGTGGCCCCTGATCCTCCGGTAGCGAAGGGCTAGGGGCACACGCCCAGC
+
BBBBFBF<FF7B########################################################################################
normal_R2
@ERR4833597.1 HISEQ:451:C78DHACXX:7:1101:10000:14561/2
GCCCTCCCTGCTCCCGGGCGGCTCTGGAGGAGCTCGGTGTTTGCCTCTAGCCCTTCGCTACCGGAGGATCAGGTTCCTCCCCTCCTCCCCCTCCAGCTCC
+
BBBFFBF<FBFFFFIF7FIFIF7FBB<0<<FBBBFBBB<'7<B<B<<B<770<BBB7<BBBBB<0000B<BBB0<<0<0707<7''007'777B<'0<BB
tumor_R1
@ERR4833621.1 HISEQ:451:C78DHACXX:1:1101:10000:16167/1
CCCACAGCAGCCCAGTGAAGAGTCCTGTTATCTTTTCGGTTTGACAGATGAGGAGATGGAGGCCAGAGAGGATGGGCCACCGGCCAAGGTCATTGTCCTG
+
BBBFFFFFFFFFFIIFIFIIIIFFIFIFFFIFFIIIIIIFIIIIIIIIIIIFIIIFFIII<FFFFFFFBFFFFFFB<BBBFBFBFFF<B0<BBFFFFFBB
tumor_R2
@ERR4833621.1 HISEQ:451:C78DHACXX:1:1101:10000:16167/2
GGGCAGTGACTGAATCCCCAAGAAAGTAGAGCCATCAAGGTCAGGGCTCTGTGGGCACAGGGTTGGGATTTAAATCCCAGCTCCACAGGACAATGACCTT
+
BBBFFFFFFFFFFIIIIIIFIIFIIIB

### Data exploration: decode a FASTQ record

The quality string is not random decoration. It encodes one quality score per DNA base. A higher score means the sequencer is more confident about that base call.

This cell prints the first record as aligned positions so you can see that sequence length and quality-string length match.

In [5]:
record = first_fastq_record(FASTQ_FILES["normal_R1"])
header, sequence, plus, quality = record

print(header)
print(f"sequence length: {len(sequence)}")
print(f"quality length:  {len(quality)}")
print()
print("First 25 bases with their ASCII quality characters:")
for pos, (base, qchar) in enumerate(zip(sequence[:25], quality[:25]), start=1):
    # Standard Sanger FASTQ encoding: Phred score = ASCII code - 33.
    phred = ord(qchar) - 33
    print(f"position {pos:2d}: base={base} quality_char={qchar!r} phred={phred}")

@ERR4833597.1 HISEQ:451:C78DHACXX:7:1101:10000:14561/1
sequence length: 100
quality length:  100

First 25 bases with their ASCII quality characters:
position  1: base=G quality_char='B' phred=33
position  2: base=G quality_char='B' phred=33
position  3: base=C quality_char='B' phred=33
position  4: base=G quality_char='B' phred=33
position  5: base=G quality_char='F' phred=37
position  6: base=C quality_char='B' phred=33
position  7: base=G quality_char='F' phred=37
position  8: base=G quality_char='<' phred=27
position  9: base=C quality_char='F' phred=37
position 10: base=G quality_char='F' phred=37
position 11: base=G quality_char='7' phred=22
position 12: base=C quality_char='B' phred=33
position 13: base=G quality_char='#' phred=2
position 14: base=G quality_char='#' phred=2
position 15: base=C quality_char='#' phred=2
position 16: base=G quality_char='#' phred=2
position 17: base=G quality_char='#' phred=2
position 18: base=C quality_char='#' phred=2
position 19: base=G quality_

### Answer 3

For the normal sample, the first `R1` and `R2` headers are:

- `@ERR4833597.1 HISEQ:451:C78DHACXX:7:1101:10000:14561/1`
- `@ERR4833597.1 HISEQ:451:C78DHACXX:7:1101:10000:14561/2`

The read identifier is the same. The suffix differs: `/1` marks `R1` and `/2` marks `R2`. That tells us these are the two mates from the same original DNA fragment.

## 4. Read Length and Read Counts

### Question 4

> What is the read length? How many reads are there per file?

The read length is the length of the sequence line in a FASTQ record. If the sequence line has 100 DNA letters, the read length is `100 bp`.

The read count is the total number of FASTQ lines divided by `4`, because each read occupies exactly four lines.

A paired-end sample has two FASTQ files. If `R1` and `R2` have the same number of reads, that means every read in one file has its mate in the other file. So the number of read pairs is the same as the read count in either mate file, not the sum of both files.

In [6]:
def fastq_stats(path):
    line_count = 0
    first_read_length = None
    with gzip.open(path, "rt") as fh:
        for line_count, line in enumerate(fh, start=1):
            if line_count == 2:
                first_read_length = len(line.rstrip())
    return {
        "read_length": first_read_length,
        "line_count": line_count,
        "read_count": line_count // 4,
    }

stats = {label: fastq_stats(path) for label, path in FASTQ_FILES.items()}

for label, result in stats.items():
    print(f"{label:10s} read_length={result['read_length']} read_count={result['read_count']:,}")

normal_R1  read_length=100 read_count=28,927,775
normal_R2  read_length=100 read_count=28,927,775
tumor_R1   read_length=100 read_count=36,383,853
tumor_R2   read_length=100 read_count=36,383,853


### Data exploration: compare read counts visually

The assignment asks for read length and read count. A quick plot helps you notice two important things:

- `R1` and `R2` have matching counts inside each sample.
- The tumor sample has more reads than the normal sample.

In [7]:
read_stats_rows = []
for label, result in stats.items():
    sample = "normal" if label.startswith("normal") else "tumor"
    mate = "R1" if label.endswith("R1") else "R2"
    read_stats_rows.append({
        "file_label": label,
        "sample": sample,
        "mate": mate,
        "read_count": result["read_count"],
        "read_length": result["read_length"],
    })

read_stats_df = pd.DataFrame(read_stats_rows)
display(read_stats_df)

fig = px.bar(
    read_stats_df,
    x="file_label",
    y="read_count",
    color="sample",
    pattern_shape="mate",
    text="read_count",
    title="Read counts per FASTQ file",
    labels={"file_label": "FASTQ file", "read_count": "reads"},
)
fig.update_traces(texttemplate="%{text:,}", textposition="outside")
fig.update_layout(yaxis_tickformat=",", xaxis_tickangle=25)
fig.show()

fig = px.bar(
    read_stats_df,
    x="file_label",
    y="read_length",
    color="sample",
    pattern_shape="mate",
    text="read_length",
    title="Read length per FASTQ file",
    labels={"file_label": "FASTQ file", "read_length": "bp"},
)
fig.update_traces(texttemplate="%{text} bp", textposition="outside")
fig.update_layout(yaxis_range=[0, 125], xaxis_tickangle=25)
fig.show()

,file_label,sample,mate,read_count,read_length
0,normal_R1,normal,R1,28927775,100
1,normal_R2,normal,R2,28927775,100
2,tumor_R1,tumor,R1,36383853,100
3,tumor_R2,tumor,R2,36383853,100


### Data exploration: sample GC content and average quality

Bioinformatics files are often too large to fully inspect by eye. A common approach is to sample the first few thousand reads and compute simple summaries.

Here we estimate two intuitive properties:

- **GC content**: fraction of bases that are `G` or `C`.
- **Mean Phred quality**: average confidence score over a read.

In [8]:
def sample_fastq_metrics(path, max_reads=5000):
    gc_values = []
    mean_quality_values = []

    with gzip.open(path, "rt") as fh:
        for read_index in range(max_reads):
            header = fh.readline()
            if not header:
                break
            sequence = fh.readline().strip().upper()
            fh.readline()  # plus line
            quality = fh.readline().strip()

            if not sequence:
                continue

            gc = (sequence.count("G") + sequence.count("C")) / len(sequence)
            phred_scores = [ord(ch) - 33 for ch in quality]
            gc_values.append(gc)
            mean_quality_values.append(sum(phred_scores) / len(phred_scores))

    return gc_values, mean_quality_values

sample_metrics = {}
sampled_rows = []
for label, path in FASTQ_FILES.items():
    gc_values, mean_quality_values = sample_fastq_metrics(path, max_reads=5000)
    sample_metrics[label] = {"gc": gc_values, "mean_quality": mean_quality_values}
    print(
        f"{label:10s} "
        f"mean_GC={statistics.mean(gc_values):.3f} "
        f"mean_read_quality={statistics.mean(mean_quality_values):.1f}"
    )
    for read_number, (gc, mean_quality) in enumerate(zip(gc_values, mean_quality_values), start=1):
        sampled_rows.append({
            "file_label": label,
            "sample": "normal" if label.startswith("normal") else "tumor",
            "mate": "R1" if label.endswith("R1") else "R2",
            "sampled_read_number": read_number,
            "gc_fraction": gc,
            "mean_quality": mean_quality,
        })

sampled_metrics_df = pd.DataFrame(sampled_rows)

fig = px.box(
    sampled_metrics_df,
    x="file_label",
    y="gc_fraction",
    color="sample",
    points=False,
    title="GC content distribution in the first 5,000 reads per file",
    labels={"file_label": "FASTQ file", "gc_fraction": "GC fraction"},
)
fig.update_layout(xaxis_tickangle=25)
fig.show()

fig = px.box(
    sampled_metrics_df,
    x="file_label",
    y="mean_quality",
    color="sample",
    points=False,
    title="Mean read quality distribution in the first 5,000 reads per file",
    labels={"file_label": "FASTQ file", "mean_quality": "mean Phred quality"},
)
fig.update_layout(xaxis_tickangle=25)
fig.show()

fig = px.scatter(
    sampled_metrics_df.sample(min(2000, len(sampled_metrics_df)), random_state=1),
    x="gc_fraction",
    y="mean_quality",
    color="file_label",
    hover_data=["sample", "mate", "sampled_read_number"],
    title="Sampled reads: GC content vs mean quality",
    labels={"gc_fraction": "GC fraction", "mean_quality": "mean Phred quality"},
)
fig.show()

normal_R1  mean_GC=0.474 mean_read_quality=37.0
normal_R2  mean_GC=0.470 mean_read_quality=36.3
tumor_R1   mean_GC=0.475 mean_read_quality=36.5
tumor_R2   mean_GC=0.472 mean_read_quality=35.8


### Answer 4

All four files have `100 bp` reads.

Read counts:

- `ERR4833597_1.fastq.gz`: `28,927,775`
- `ERR4833597_2.fastq.gz`: `28,927,775`
- `ERR4833621_1.fastq.gz`: `36,383,853`
- `ERR4833621_2.fastq.gz`: `36,383,853`

Because this is paired-end data, the normal sample has `28,927,775` read pairs and the tumor sample has `36,383,853` read pairs.

## 5. Run FastQC Before Quality Questions

The README asks us to run FastQC before answering the next two questions.

> Run FastQC on all the files. Did you get the number of reads per file correct?

FastQC is a quality-control tool for raw sequencing files. It does not solve the biological question; it tells us whether the input data looks technically reasonable.

FastQC produces HTML reports and `.zip` archives. The zip archives contain `summary.txt` and `fastqc_data.txt`, which are easy to parse programmatically.

If FastQC has already been run, this cell can be skipped. It is safe to run again; it will overwrite the reports.

In [9]:
FASTQC_DIR.mkdir(exist_ok=True)

cmd = [
    str(Path.home() / "anaconda3/bin/conda"),
    "run", "-n", "bioinfo",
    "fastqc", "-t", "4", "-o", str(FASTQC_DIR),
    *[str(path) for path in FASTQ_FILES.values()],
]

print(" ".join(cmd))
# Uncomment the next line if you need to run FastQC from the notebook.
# subprocess.run(cmd, check=True)

/home/orgr/anaconda3/bin/conda run -n bioinfo fastqc -t 4 -o /home/orgr/compgenomicsws_hw2/fastqc /home/orgr/compgenomicsws_hw2/ERR4833597_1.fastq.gz /home/orgr/compgenomicsws_hw2/ERR4833597_2.fastq.gz /home/orgr/compgenomicsws_hw2/ERR4833621_1.fastq.gz /home/orgr/compgenomicsws_hw2/ERR4833621_2.fastq.gz


## 6. FastQC Quality Interpretation

The summary tells us whether each QC module passed, warned, or failed.

### Question 5

> Check the `Per base sequence quality`. Do the overall base qualities look ok? Do you notice any strange behavior?

### Question 6

> Does the `Per base sequence content` behave as expected?

Plain-English meanings:

- **Per base sequence quality**: are the letters reliable at each position along the read?
- **Per base sequence content**: at each position, do the proportions of `A`, `C`, `G`, and `T` look reasonable?

A warning is not automatically a disaster. Real sequencing data often has small warnings, especially near the start or end of reads.

In [10]:
def read_fastqc_summary(zip_path):
    with zipfile.ZipFile(zip_path) as zf:
        summary_name = next(name for name in zf.namelist() if name.endswith("summary.txt"))
        text = zf.read(summary_name).decode()
    return [line.split("\t") for line in text.strip().splitlines()]

for zip_path in sorted(FASTQC_DIR.glob("*_fastqc.zip")):
    print("=" * 80)
    print(zip_path.name)
    for status, module, filename in read_fastqc_summary(zip_path):
        if module in {"Basic Statistics", "Per base sequence quality", "Per base sequence content", "Per tile sequence quality", "Per sequence GC content"}:
            print(f"{status:4s} {module}")

ERR4833597_1_fastqc.zip
PASS Basic Statistics
PASS Per base sequence quality
PASS Per tile sequence quality
PASS Per base sequence content
WARN Per sequence GC content
ERR4833597_2_fastqc.zip
PASS Basic Statistics
PASS Per base sequence quality
PASS Per tile sequence quality
PASS Per base sequence content
WARN Per sequence GC content
ERR4833621_1_fastqc.zip
PASS Basic Statistics
PASS Per base sequence quality
WARN Per tile sequence quality
PASS Per base sequence content
WARN Per sequence GC content
ERR4833621_2_fastqc.zip
PASS Basic Statistics
PASS Per base sequence quality
WARN Per tile sequence quality
PASS Per base sequence content
WARN Per sequence GC content


In [11]:
def extract_fastqc_module(zip_path, module_name):
    with zipfile.ZipFile(zip_path) as zf:
        data_name = next(name for name in zf.namelist() if name.endswith("fastqc_data.txt"))
        lines = zf.read(data_name).decode().splitlines()

    capture = False
    module_lines = []
    for line in lines:
        if line.startswith(f">>{module_name}"):
            capture = True
        if capture:
            module_lines.append(line)
        if capture and line == ">>END_MODULE":
            break
    return module_lines

for zip_path in sorted(FASTQC_DIR.glob("*_fastqc.zip")):
    print("=" * 80)
    print(zip_path.name)
    for module in ["Basic Statistics", "Per base sequence quality", "Per base sequence content"]:
        lines = extract_fastqc_module(zip_path, module)
        print(lines[0])
    print()

ERR4833597_1_fastqc.zip
>>Basic Statistics	pass
>>Per base sequence quality	pass
>>Per base sequence content	pass

ERR4833597_2_fastqc.zip
>>Basic Statistics	pass
>>Per base sequence quality	pass
>>Per base sequence content	pass

ERR4833621_1_fastqc.zip
>>Basic Statistics	pass
>>Per base sequence quality	pass
>>Per base sequence content	pass

ERR4833621_2_fastqc.zip
>>Basic Statistics	pass
>>Per base sequence quality	pass
>>Per base sequence content	pass



### Data exploration: plot FastQC quality and base composition

The HTML FastQC reports contain plots, but we can also parse the underlying text. This makes the assignment more transparent: the plot is just numbers in `fastqc_data.txt`.

The next cell plots:

- mean quality by read position
- base composition by read position for one representative file

In [12]:
def module_table(zip_path, module_name):
    lines = extract_fastqc_module(zip_path, module_name)
    rows = []
    header = None
    for line in lines:
        if line.startswith(">>"):
            continue
        if line.startswith("#"):
            header = line[1:].split("	")
            continue
        rows.append(line.split("	"))
    return header, rows


def parse_position_range(value):
    # FastQC sometimes uses ranges such as "10-14". Use the midpoint for plotting.
    if "-" in value:
        start, end = value.split("-", 1)
        return (float(start) + float(end)) / 2
    return float(value)

fastqc_zip_files = sorted(FASTQC_DIR.glob("*_fastqc.zip"))
if not fastqc_zip_files:
    raise FileNotFoundError(
        f"No FastQC zip files found in {FASTQC_DIR}. Run the FastQC cell first, "
        "or check that DATA_DIR points to the downloaded homework data."
    )

quality_rows = []
for zip_path in fastqc_zip_files:
    header, rows = module_table(zip_path, "Per base sequence quality")
    base_i = header.index("Base")
    mean_i = header.index("Mean")
    lower_i = header.index("Lower Quartile")
    upper_i = header.index("Upper Quartile")
    for row in rows:
        quality_rows.append({
            "file": zip_path.name.replace("_fastqc.zip", ""),
            "position": parse_position_range(row[base_i]),
            "mean_quality": float(row[mean_i]),
            "lower_quartile": float(row[lower_i]),
            "upper_quartile": float(row[upper_i]),
        })

quality_df = pd.DataFrame(quality_rows)
fig = px.line(
    quality_df,
    x="position",
    y="mean_quality",
    color="file",
    hover_data=["lower_quartile", "upper_quartile"],
    title="FastQC mean quality by read position",
    labels={"position": "position in read", "mean_quality": "mean Phred quality"},
)
fig.show()

# Plot base composition for one representative file to keep the figure readable.
representative_zip = FASTQC_DIR / "ERR4833597_1_fastqc.zip"
if not representative_zip.exists():
    representative_zip = fastqc_zip_files[0]
header, rows = module_table(representative_zip, "Per base sequence content")
base_i = header.index("Base")
composition_rows = []
for row in rows:
    position = parse_position_range(row[base_i])
    for base in ["G", "A", "T", "C"]:
        composition_rows.append({
            "file": representative_zip.name.replace("_fastqc.zip", ""),
            "position": position,
            "base": base,
            "percent": float(row[header.index(base)]),
        })

composition_df = pd.DataFrame(composition_rows)
fig = px.line(
    composition_df,
    x="position",
    y="percent",
    color="base",
    title=f"Base composition by position: {representative_zip.name}",
    labels={"position": "position in read", "percent": "percent of bases"},
)
fig.show()

### Answer 5

Overall base quality looks good. All four files pass `Per base sequence quality`.

There is the usual quality drop toward the 3' end, especially in `R2` files. This is common because sequencing confidence often decreases later in the read. There are also visible dips around cycles `50-53` in `R2` and around `76-77` in `R1`.

### Answer 6

`Per base sequence content` passes in all four files. The beginning of the reads is not perfectly flat: the first roughly `10-15` bases show composition bias, then the base proportions stabilize.

That early-cycle bias is common in real sequencing libraries and can reflect priming or library-preparation effects. So the result is not perfectly ideal-looking, but it is acceptable and passes FastQC.

## 7. Reference Genome Questions

The README next asks us to inspect the downloaded human reference genome FASTA file, `hg38.fa.gz`.

### Question 7

> What is `hg38`?

### Question 8

> How many sequences are present in the human reference genome you just downloaded?

### Question 9

> Just by looking at the sequence names can you understand what sequences they are?

### Question 10

> How many nucleotides are there in chromosome 1? How many in chromosome 20?

`hg38` is the UCSC name for the `GRCh38` human reference genome assembly. In practical terms, this is the version of the human genome we use as a coordinate system.

FASTA files store each sequence with a header beginning with `>`, followed by nucleotide sequence lines. Counting headers gives the number of sequences in the reference.

Important: the reference genome is not stored as one giant string. It is stored as many named sequences: main chromosomes like `chr1`, sex chromosomes like `chrX`, and extra scaffolds/contigs that represent pieces that are harder to place cleanly.

In [13]:
def fasta_lengths(path):
    lengths = {}
    current_name = None
    current_length = 0

    with gzip.open(path, "rt") as fh:
        for line in fh:
            if line.startswith(">"):
                if current_name is not None:
                    lengths[current_name] = current_length
                current_name = line[1:].split()[0]
                current_length = 0
            else:
                current_length += len(line.strip())

    if current_name is not None:
        lengths[current_name] = current_length

    return lengths

lengths = fasta_lengths(HG38)
print(f"Number of sequences: {len(lengths)}")
print(f"chr1 length:  {lengths['chr1']:,}")
print(f"chr20 length: {lengths['chr20']:,}")

Number of sequences: 455
chr1 length:  248,956,422
chr20 length: 64,444,167


In [14]:
names = list(lengths)
print("First 25 sequence names:")
for name in names[:25]:
    print(name)

print("\nLast 25 sequence names:")
for name in names[-25:]:
    print(name)

First 25 sequence names:
chr1
chr10
chr11
chr11_KI270721v1_random
chr12
chr13
chr14
chr14_GL000009v2_random
chr14_GL000225v1_random
chr14_KI270722v1_random
chr14_GL000194v1_random
chr14_KI270723v1_random
chr14_KI270724v1_random
chr14_KI270725v1_random
chr14_KI270726v1_random
chr15
chr15_KI270727v1_random
chr16
chr16_KI270728v1_random
chr17
chr17_GL000205v2_random
chr17_KI270729v1_random
chr17_KI270730v1_random
chr18
chr19

Last 25 sequence names:
chrUn_KI270741v1
chrUn_GL000226v1
chrUn_GL000213v1
chrUn_KI270743v1
chrUn_KI270744v1
chrUn_KI270745v1
chrUn_KI270746v1
chrUn_KI270747v1
chrUn_KI270748v1
chrUn_KI270749v1
chrUn_KI270750v1
chrUn_KI270751v1
chrUn_KI270752v1
chrUn_KI270753v1
chrUn_KI270754v1
chrUn_KI270755v1
chrUn_KI270756v1
chrUn_KI270757v1
chrUn_GL000214v1
chrUn_KI270742v1
chrUn_GL000216v2
chrUn_GL000218v1
chrX
chrY
chrY_KI270740v1_random


### Data exploration: chromosome and contig lengths

The reference contains 455 named sequences, but they are not all equally important or equally large. Plotting lengths makes the structure obvious: the main chromosomes dominate, while many auxiliary contigs are much smaller.

In [15]:
main_chromosomes = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]
main_lengths_df = pd.DataFrame({
    "sequence": main_chromosomes,
    "length": [lengths[name] for name in main_chromosomes],
})

fig = px.bar(
    main_lengths_df,
    x="sequence",
    y="length",
    text="length",
    title="Main hg38 chromosome lengths",
    labels={"sequence": "chromosome", "length": "nucleotides"},
)
fig.update_traces(texttemplate="%{text:,}", textposition="outside")
fig.update_layout(yaxis_tickformat=",", xaxis_tickangle=45)
fig.show()

all_lengths_df = pd.DataFrame(
    [{"sequence": name, "length": length} for name, length in lengths.items()]
)

def classify_sequence(name):
    if name in set(main_chromosomes):
        return "main chromosome"
    if name.startswith("chrUn_"):
        return "unplaced contig"
    if "_random" in name:
        return "random/unlocalized scaffold"
    if "_alt" in name:
        return "alternate contig"
    return "other auxiliary sequence"

all_lengths_df["sequence_type"] = all_lengths_df["sequence"].map(classify_sequence)
all_lengths_df = all_lengths_df.sort_values("length", ascending=False)

fig = px.histogram(
    all_lengths_df,
    x="length",
    color="sequence_type",
    log_x=True,
    title="Distribution of hg38 sequence lengths",
    labels={"length": "sequence length, log scale", "count": "number of sequences"},
)
fig.show()

fig = px.bar(
    all_lengths_df.head(30),
    x="sequence",
    y="length",
    color="sequence_type",
    title="Thirty longest hg38 sequences",
    labels={"sequence": "sequence name", "length": "nucleotides"},
)
fig.update_layout(yaxis_tickformat=",", xaxis_tickangle=45)
fig.show()

print("Ten longest sequences:")
for name, length in all_lengths_df[["sequence", "length"]].head(10).itertuples(index=False):
    print(f"{name:12s} {length:,}")

print("\nTen shortest sequences:")
for name, length in all_lengths_df[["sequence", "length"]].tail(10).itertuples(index=False):
    print(f"{name:20s} {length:,}")

Ten longest sequences:
chr1         248,956,422
chr2         242,193,529
chr3         198,295,559
chr4         190,214,555
chr5         181,538,259
chr6         170,805,979
chr7         159,345,973
chrX         156,040,895
chr8         145,138,636
chr9         138,394,717

Ten shortest sequences:
chrUn_KI270379v1     1,045
chrUn_KI270329v1     1,040
chrUn_KI270419v1     1,029
chrUn_KI270336v1     1,026
chrUn_KI270312v1     998
chrUn_KI270539v1     993
chrUn_KI270385v1     990
chrUn_KI270423v1     981
chrUn_KI270392v1     971
chrUn_KI270394v1     970


### Answer 7

`hg38` is UCSC's name for the human reference genome assembly `GRCh38`. It is a standard reference coordinate system for human DNA.

### Answer 8

The downloaded `hg38.fa.gz` contains `455` sequences.

### Answer 9

The sequence names are understandable at a high level:

- `chr1`-`chr22`, `chrX`, and `chrY` are the main chromosomes.
- Names with `_random` are unlocalized/random scaffolds, meaning we know roughly which chromosome they belong to but not their exact position.
- Names starting with `chrUn_` are unplaced contigs, meaning they are known human DNA sequence but not assigned to a chromosome position.
- Names containing `KI...` or `GL...` are accession-based auxiliary or alternate contigs.

### Answer 10

- `chr1` has `248,956,422` nucleotides.
- `chr20` has `64,444,167` nucleotides.

## 8. How I Approached the Task

When a genomics assignment feels unfamiliar, I treat it like a data-engineering problem first:

1. Find the assignment files and identify the inputs.
2. Read the metadata to map sample names to biological meaning.
3. Inspect one tiny piece of each large file before trying to process everything.
4. Use file-format rules to compute simple facts: four FASTQ lines per read, one FASTA header per sequence.
5. Run a standard QC tool, then parse its output rather than relying only on screenshots.
6. Convert final observations into short biological interpretations.

That approach keeps the biology manageable: first understand the data objects, then attach the biological names to them.